# Parts 2–4 · Run a Jev virtual-tissue simulation

| Part | In this notebook | ✏️ What you set |
|---|---|---|
| **2 · Input** | load the bundle ZIP, verify it, look at the measured tissue | `INPUT_SOURCE`, `BUNDLE_ZIP`, results folder |
| **3 · Run** | scenario, duration, seed, API limits, preflight estimate, Jev key, run | `SCENARIO`, `HORIZON_HOURS`, `MAX_HTTP_ATTEMPTS`, … , `START_RUN` |
| **4 · Results** | interactive replay, summary figures, download | — |

**You need** a prepared bundle ZIP (`simulation_input_<REGION>_<time>.zip`) from Part 1, and for real runs a
**Jev (TypeSafe) API key** stored in Colab **Secrets** as `TYPESAFE_API_KEY` (or `JEV_API_KEY`).
Without a key you can still run the free **local preview** (programmed rules, no API).

**Colab runtime:** Python 3 · CPU. Run the cells in order. Guides:
[Part 2](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/02_part2_input_bundle.md) ·
[Part 3](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/03_part3_run_simulation.md) ·
[Part 4](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/04_part4_results.md) ·
[Jev key](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/07_jev_api_key.md)

The experiment is an **uncalibrated mechanism simulation**, not a patient prediction. Measured t0 RNA scores
inform local choices; fields, kinetic rates, recognition and effector competence are assumptions. There is no
automatic fallback from a failed Jev call to the local policy.


## 1 · Get the code and install dependencies
If a restart is requested, choose **Runtime → Restart session** and rerun this cell. No Jev request is sent here.


In [ ]:
#@title Get the VirtualTissue-Xenium code and install the pinned dependencies
# ✏️ EDIT only if needed — where the code comes from:
#   github        clone the public repository (default)
#   google_drive  copy the VirtualTissue-Xenium folder you placed in Google Drive
#   installed     the package is already installed with pip (local Jupyter)
CODE_SOURCE = "github"  #@param ["github", "google_drive", "installed"]
GITHUB_URL = "https://github.com/hylixinsights/VirtualTissue-Xenium.git"  #@param {type:"string"}
GITHUB_BRANCH = "main"  #@param {type:"string"}
DRIVE_CODE_DIR = "/content/drive/MyDrive/VirtualTissue-Xenium"  #@param {type:"string"}
INSTALL_DEPENDENCIES = True  #@param {type:"boolean"}

import importlib, os, shutil, subprocess, sys
from pathlib import Path
print("Running Python:", sys.version.split()[0], "| executable:", sys.executable)
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# Temporary Colab disk. Keep the code here, not in Drive.
CODE_DIR = Path("/content/VirtualTissue-Xenium")
if not IN_COLAB and CODE_SOURCE != "installed":
    # Local Jupyter: use the repository that contains this notebook.
    here = Path.cwd().resolve()
    CODE_DIR = here.parent if (here.parent / "src" / "virtualtissue_xenium").is_dir() else here
    CODE_SOURCE = "local_repository"
if CODE_SOURCE == "github":
    if not (CODE_DIR / "src" / "virtualtissue_xenium").is_dir():
        subprocess.check_call(["git", "clone", "--quiet", "--depth", "1", "--branch", GITHUB_BRANCH,
                               GITHUB_URL, str(CODE_DIR)])
elif CODE_SOURCE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not (Path(DRIVE_CODE_DIR) / "src" / "virtualtissue_xenium").is_dir():
        raise FileNotFoundError("DRIVE_CODE_DIR must be the VirtualTissue-Xenium folder "
                                "(the one that contains src/, notebooks/ and README.md).")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, CODE_DIR,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".ipynb_checkpoints"))
if CODE_SOURCE != "installed" and str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))
import virtualtissue_xenium as vtx
from virtualtissue_xenium import colab as vtx_colab
print("VirtualTissue-Xenium", vtx.__version__, "| engine:", vtx.ENGINE_DIR)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/vtx_matplotlib")


import colab_setup
importlib.reload(colab_setup)
SETUP = colab_setup.setup_environment(vtx.ENGINE_DIR, install_packages=INSTALL_DEPENDENCIES)
SETUP_READY = SETUP["ready"]

if not SETUP_READY:
    raise RuntimeError("Restart the session and rerun this setup cell before continuing.")
import simulation_workflow as workflow
print("Setup complete. Simulation engine:", workflow.VERSION, "| No Jev requests have been sent.")


# PART 2 · Provide and verify the input
The input is **one bundle ZIP** produced by Part 1 (`export_region` → `zip_bundle`). Keep it compressed. It
contains the region's cells (≤ 1,000 including context), their polygons, contacts, provisional identities and
t0 AUCell evidence, plus the runtime contract and checksums. The simulator verifies every checksum and refuses
any bundle whose embedded runtime differs from this engine; Python files inside the ZIP are never executed.
Using your own data? Run Part 1 on your Xenium output first —
see [docs/06_your_own_xenium_data.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/06_your_own_xenium_data.md).

✏️ **INPUT_SOURCE**
* `google_drive` — `BUNDLE_ZIP` is a path in Drive. It may be the ZIP itself or the Part 1 `run_…` folder
  (then the newest `simulation_input_*.zip` inside is used). **To resume, choose the exact original ZIP**
  so a later export is not selected by mistake.
* `upload` — a file chooser appears; pick the ZIP on your computer.
* `url` — download the ZIP from `BUNDLE_URL` (for example a GitHub release asset).

✏️ **SAVE_RESULTS_TO_DRIVE** keeps a copy of the run in `DRIVE_RESULTS_DIR`, refreshed after every simulated
hour and at the end. If the runtime is recycled, rerun with the **same input ZIP, RUN_NAME, settings and
software environment**: the latest successful copy is restored and the responses it contains are reused.
Paid answers since that backup may be lost and requested again. Before the first successful backup there
may be no run to restore; failed backups can extend the gap. Saving this notebook does not save session files.

**Local Jupyter:** keep `INPUT_SOURCE = google_drive` (here it means a file path), replace `BUNDLE_ZIP`
with your local ZIP or output folder, and **uncheck `SAVE_RESULTS_TO_DRIVE`** or change `DRIVE_RESULTS_DIR`
to your local backup folder. The `/content/drive/...` defaults are not converted automatically.
The `upload` chooser requires Colab. Working files are stored in `vtx_session/`.


In [ ]:
#@title ✏️ Input bundle and results folder
INPUT_SOURCE = "google_drive"  #@param ["google_drive", "upload", "url"]
BUNDLE_ZIP = "/content/drive/MyDrive/VirtualTissue/Xenium_Lung5K_Prepared"  #@param {type:"string"}
BUNDLE_URL = ""  #@param {type:"string"}
SAVE_RESULTS_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/VirtualTissue/Simulation_Results"  #@param {type:"string"}

import urllib.request
# Working folders on the temporary session disk (fast, local). Drive keeps the copies.
SESSION_ROOT = Path("/content/Xenium_Jev") if IN_COLAB else Path("vtx_session").resolve()
INPUT_DIR = SESSION_ROOT / "inputs"
RESULT_ROOT = SESSION_ROOT / "results"
for folder in (INPUT_DIR, RESULT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
if IN_COLAB and (SAVE_RESULTS_TO_DRIVE or INPUT_SOURCE == "google_drive"):
    vtx_colab.mount_drive()
if INPUT_SOURCE == "google_drive":
    source = Path(BUNDLE_ZIP)
    if source.is_dir():
        candidates = sorted(source.rglob("simulation_input_*.zip"), key=lambda p: p.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError(f"No simulation_input_*.zip under {source}.")
        source = candidates[-1]
        print("Newest bundle in that folder:", source)
    if not source.is_file() or source.suffix != ".zip":
        raise FileNotFoundError(f"BUNDLE_ZIP must be a .zip file or a folder that contains one: {source}")
    INPUT_ZIP = INPUT_DIR / source.name
    if not INPUT_ZIP.is_file() or INPUT_ZIP.stat().st_size != source.stat().st_size:
        shutil.copyfile(source, INPUT_ZIP)
elif INPUT_SOURCE == "upload":
    from google.colab import files
    previous_directory = Path.cwd()
    try:
        os.chdir(INPUT_DIR)
        uploaded = files.upload()
    finally:
        os.chdir(previous_directory)
    if len(uploaded) != 1:
        raise ValueError("Select exactly one prepared simulation ZIP.")
    name, content = next(iter(uploaded.items()))
    if not name.lower().endswith(".zip"):
        raise ValueError("Select the simulation_input ZIP file.")
    INPUT_ZIP = INPUT_DIR / Path(name).name
    INPUT_ZIP.write_bytes(content)
    del content, uploaded
elif INPUT_SOURCE == "url":
    if not BUNDLE_URL.startswith("https://"):
        raise ValueError("BUNDLE_URL must be an https:// link to a simulation_input ZIP.")
    INPUT_ZIP = INPUT_DIR / Path(BUNDLE_URL.split("?")[0]).name
    if not INPUT_ZIP.is_file():
        urllib.request.urlretrieve(BUNDLE_URL, INPUT_ZIP)
else:
    raise ValueError("Unknown INPUT_SOURCE.")
BACKUP_ROOT = Path(DRIVE_RESULTS_DIR) if SAVE_RESULTS_TO_DRIVE else None
BUNDLE = workflow.extract_bundle(INPUT_ZIP, SESSION_ROOT / "extracted")
print("Verified:", BUNDLE["atlas"]["patch_id"])
print("Input SHA-256:", BUNDLE["zip_sha256"])
print("Working copy of the input:", INPUT_ZIP)
print("Results (session disk):", RESULT_ROOT)
print("Results copy in Drive:", BACKUP_ROOT or "disabled — download the results ZIP in Part 4")


### What is inside the bundle
`atlas.json.gz` holds one record per cell: `role` (`roi` = acting cell, `occluded` = context inside the region,
`halo` = surrounding context), polygon `geometry`, `contacts` (polygon distance ≤ 1 µm), `identity`
(provisional label) and `signatures` (20 AUCell scores with within-type/depth percentiles).
`region.h5ad` holds the raw counts and XY for the same cells. Full schema:
[docs/02_part2_input_bundle.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/02_part2_input_bundle.md).


In [ ]:
#@title Inspect the bundle and the measured t0 tissue (no API calls)
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Patch
from shapely.geometry import shape
from IPython.display import display
atlas = BUNDLE["atlas"]
roles = Counter(c["role"] for c in atlas["cells"])
print(f"{atlas['patch_id']}: {len(atlas['cells'])} cells · roles {dict(roles)}")
print("Tumour handling:", atlas["provenance"].get("tumor_mode"), "| score method:", atlas["provenance"].get("score_method"))
print("Signatures (t0 AUCell):", ", ".join(sorted(atlas["signatures"])))
files = json.loads((BUNDLE["folder"] / "BUNDLE_MANIFEST.json").read_text())["files"]
display(pd.DataFrame([dict(file=k, kilobytes=round(v["bytes"] / 1024, 1)) for k, v in files.items()
                      if not k.startswith("xenium_runtime/")]).set_index("file"))
acting = Counter((c.get("identity") or {}).get("candidates", [{}])[0].get("label") for c in atlas["cells"] if c["role"] == "roi")
display(pd.DataFrame(sorted(acting.items()), columns=["acting cell type", "cells"]).set_index("acting cell type"))
labels = sorted(acting)
colors = {label: plt.get_cmap("tab20")(i / max(1,len(labels)-1)) for i,label in enumerate(labels)}
fig, ax = plt.subplots(figsize=(10,9))
for cell in atlas["cells"]:
    active = cell["role"] == "roi"
    identity = cell.get("identity") or {}
    label = (identity.get("candidates") or [{"label":"context"}])[0]["label"]
    geometry = shape(cell["geometry"])
    parts = [geometry] if geometry.geom_type == "Polygon" else geometry.geoms
    for polygon in parts:
        from shapely.geometry.polygon import orient
        polygon = orient(polygon, sign=1.0)
        rings = [polygon.exterior, *polygon.interiors]
        vertices, codes = [], []
        for ring in rings:
            points = list(ring.coords)
            vertices.extend(points)
            codes.extend([MplPath.MOVETO] + [MplPath.LINETO]*(len(points)-2) + [MplPath.CLOSEPOLY])
        ax.add_patch(PathPatch(MplPath(vertices,codes), facecolor=colors.get(label,"#999999"),
                              alpha=.85 if active else .22, edgecolor="white", linewidth=.15))
ax.autoscale_view(); ax.set_aspect("equal"); ax.invert_yaxis()
ax.set(xlabel="X (µm)", ylabel="Y (µm)", title=f"{atlas['patch_id']} · measured segmentation · provisional actor labels")
ax.legend(handles=[Patch(color=colors[label],label=label) for label in labels] + [Patch(color="#cccccc", label="Context")],
          loc="upper left", bbox_to_anchor=(1.01,1), frameon=False)
plt.tight_layout(); plt.show()
MEASURED_T0_FIGURE = fig


# PART 3 · Configure and run the simulation

**To test CD8-mediated tumour killing, explicitly select `declared_recognition`.** Compare with
`recognition_off` using the same region, seed, model, duration and population ceiling. The default
`baseline_unknown` does not enable immune killing, even with a longer horizon. Hypoxic necrosis is separate.

**Scenario**
* `baseline_unknown` — recognition and effector/clearance competence stay unknown; cytotoxic killing is not licensed.
* `declared_recognition` — assume antigen recognition and effector/clearance competence. Enables contact-dependent
  actions; it does not guarantee killing or infer recognition from RNA.
* `recognition_off` — same declared competences with recognition explicitly false (negative control).

**Limits** (the run stops at whichever comes first, and still saves a valid replay up to that point)
* `HORIZON_HOURS` — simulated time (multiple of 10 min, max 7 days). Decisions are checkpointed every 10 min.
* `MAX_HTTP_ATTEMPTS` — every POST to Jev counts, including failures, across resumptions. Each POST carries
  up to 20 cell questions (≤ 40 KB). **This is your cost control.** It is not a dollar cap.
* `MAX_CELL_DECISIONS` — total cell questions; if a round would exceed it, a seeded subset is asked.
* `MAX_POPULATION` — body ceiling including dead/removed bodies (default 1,000). The starting region already
  uses most of it (e.g. R03: 989 → only 11 new cells). Raise it to study growth.

`RUN_NAME` names the results folder; for your own sample, use a name such as `MySample_R01_jev01`.
Keep the name and every setting unchanged to **resume**; use a new name
for a different scenario, duration, tissue or seed. With its declared lung tissue, R03 asks 280 questions in
25 requests at t0; use your preflight count. An earlier 24 h validation run with mocked answers needed
122 requests. Real Jev choices may need more, so start with a
small cap and resume with a larger one (saved answers are reused).


In [ ]:
#@title ✏️ Experiment settings
RUN_NAME = "R03_jev_run01" #@param {type:"string"}
RUN_MODE = "Jev" #@param ["Jev", "Local preview (no API)"]
SCENARIO = "baseline_unknown" #@param ["baseline_unknown", "declared_recognition", "recognition_off"]
JEV_MODEL = "jev-1.14.0" #@param ["jev-1.14.0", "jev-1.13.0"]
TISSUE = "" #@param {type:"string"}
# Empty uses SOURCE.json; set this for older own-data bundles without a tissue declaration.
HORIZON_HOURS = 24 #@param {type:"number"}
SEED = 21 #@param {type:"integer"}
MAX_POPULATION = 1000 #@param {type:"integer"}
MAX_HTTP_ATTEMPTS = 600 #@param {type:"integer"}
MAX_CELL_DECISIONS = 10000 #@param {type:"integer"}
import re
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", RUN_NAME):
    raise ValueError("RUN_NAME must use letters, numbers, underscores, or hyphens.")
if type(MAX_HTTP_ATTEMPTS) is not int or MAX_HTTP_ATTEMPTS < 1:
    raise ValueError("The HTTP limit must be a positive integer.")
# Whole hours are stored as integers so that the run configuration (and resuming) is stable.
horizon_hours = int(HORIZON_HOURS) if float(HORIZON_HOURS).is_integer() else HORIZON_HOURS
SETTINGS = dict(model=JEV_MODEL, scenario=SCENARIO, seed=SEED, horizon_hours=horizon_hours,
                max_decisions=MAX_CELL_DECISIONS, max_population=MAX_POPULATION, tissue=TISSUE or None)
MODE = "jev" if RUN_MODE == "Jev" else "local_preview"
RUN_DIR = RESULT_ROOT / (RUN_NAME + ("_local_preview" if MODE == "local_preview" else ""))
workflow.effective_contract(BUNDLE, SETTINGS)
print("Output folder:", RUN_DIR)
print("Run mode:", RUN_MODE)


### Preflight: check before spending API credits
Makes **zero API calls**: one static mechanical step, the number of questions and HTTP requests of the
**first** round, and a sample request saved as JSON so you can see exactly what Jev receives. Later rounds
depend on the trajectory, so this is not a forecast of the total.

Each question contains the cell's permitted menu, its t0 raw AUCell values and percentiles (whole-sample
type + detection-depth reference; missing scores stay missing), local contacts and simulated state.


In [ ]:
#@title Preflight estimate and request preview (no API calls)
PREFLIGHT, FIRST_REQUEST = workflow.preflight(BUNDLE, SETTINGS)
from virtualtissue_xenium.experiments import experiment_notes
for note in experiment_notes(SETTINGS, PREFLIGHT):
    print(note)
display(pd.DataFrame([dict(cell_type=k, actors=v) for k,v in PREFLIGHT["acting_composition"].items()]))
print(json.dumps({k:PREFLIGHT[k] for k in ("total_cells", "acting_cells", "context_cells", "initial_cell_questions", "initial_http_batches", "growth_headroom", "static_preflight_passed", "t0_activity", "prompt_context")}, indent=2))
if MAX_HTTP_ATTEMPTS < PREFLIGHT["initial_http_batches"] and MODE == "jev":
    print("The allowance cannot cover the first round. Increase it before starting; otherwise no POST is sent.")
PREVIEW_DIR = RESULT_ROOT / "preflight" / RUN_NAME
PREVIEW_DIR.mkdir(parents=True,exist_ok=True)
workflow.write_json(PREVIEW_DIR / "preflight.json", PREFLIGHT)
workflow.write_json(PREVIEW_DIR / "first_request_preview.json", FIRST_REQUEST)
MEASURED_T0_FIGURE.savefig(PREVIEW_DIR / "measured_t0.png", dpi=160, bbox_inches="tight")
print("Request preview:", PREVIEW_DIR / "first_request_preview.json")
if FIRST_REQUEST:
    q = next(iter(FIRST_REQUEST["questions"].values()))
    print("Example question — permitted options:", list(q["criteria"]))


### Add your Jev key and start
1. Click the **key icon (Secrets)** in Colab's left sidebar → **Add new secret** → Name `TYPESAFE_API_KEY`
   (or `JEV_API_KEY`) → paste the key as Value → switch **Notebook access** on.
   Locally, set the environment variable `TYPESAFE_API_KEY` instead. If no key is found, a hidden prompt appears.
   **Never paste a key into a code cell.** The key is not written to the journal, replay or results.
2. ✏️ Check **START_RUN** and run the cell. Checking it authorises spending up to `MAX_HTTP_ATTEMPTS` requests.

Every Jev batch is journaled before sending. There are **no automatic retries**. A whole round is validated before
any decision is applied; API errors, malformed answers or model drift stop the run, which is still saved.
Checkpoints are written every simulated hour. To resume after an interruption, rerun the notebook from the top
with the same input ZIP, `RUN_NAME`, settings and software environment. The engine replays from t0 using
the responses in the available journal and sends only missing batches. After runtime loss, responses absent
from the last successful Drive copy may be requested and paid for again. An interrupted request whose
entry survives in the journal has unknown billing status and is never resent automatically.


In [ ]:
#@title Optional: unlock after a terminated session
# Leave false during a normal run. Use only after stopping all other sessions writing this run.
CONFIRM_PREVIOUS_SESSION_STOPPED = False #@param {type:"boolean"}
if CONFIRM_PREVIOUS_SESSION_STOPPED:
    (RUN_DIR / ".running.lock").unlink(missing_ok=True)
    print("Stale lock removed. Existing journal and checkpoints were preserved.")


In [ ]:
#@title ✏️ Start or resume the simulation
START_RUN = False #@param {type:"boolean"}
def start_selected_run():
    if not START_RUN:
        print("Ready. Review the scenario and limits, then check START_RUN and rerun this cell.")
        return None
    key = ""
    try:
        if MODE == "jev":
            key = vtx_colab.get_jev_key()
            if not key:
                raise ValueError("No API key provided. No Jev request was sent.")
        if BACKUP_ROOT and vtx_colab.restore_run(BACKUP_ROOT, RUN_DIR):
            print("Restored this run from", BACKUP_ROOT / RUN_DIR.name, "- responses in this backup will be reused.")
        progress = vtx_colab.progress_with_backup(RUN_DIR, BACKUP_ROOT)
        return workflow.run(BUNDLE, SETTINGS, RUN_DIR, api_key=key,
                            max_http_attempts=MAX_HTTP_ATTEMPTS, mode=MODE, progress=progress)
    finally:
        key = ""
RESULT = start_selected_run()
if RESULT:
    print(json.dumps(RESULT["report"], indent=2))


# PART 4 · Explore and keep the results

The replay supports a time slider, play/pause, cell types, current actions, t0 AUCell percentiles, simulated
field overlays (IFN-γ, CXCL9, O₂, TGF-β) and a cell inspector with each cell's decision history and Jev
probabilities. The t0 view uses measured boundaries; later shapes and smooth motion render the recorded
simulated states. Full guide: [docs/04_part4_results.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/04_part4_results.md).

The results ZIP contains `replay.html` (opens offline in any browser), `episode.json.gz`, the Jev journal,
`metrics.csv` / `events.csv` / `decisions.csv`, the input provenance, the effective contract and a run report.


In [ ]:
#@title Interactive replay
report_path = RUN_DIR / "run_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("Completed:", report["completed"], "| Stop reason:", report["stop_reason"])
    print("Saved simulation hours:", report["checkpoint_time_min"] / 60)
    print("API usage:", report["usage"])
    if report.get("provider_error"):
        print("Provider:", report["provider_error"])
    vtx_colab.show_replay(RUN_DIR / "replay.html")
else:
    print("Run the simulation cell first. No saved result is available for this RUN_NAME.")


In [ ]:
#@title Summary figures and tables
from virtualtissue_xenium import results as vtx_results
if report_path.exists():
    RES = vtx_results.load_results(RUN_DIR)
    display(pd.Series(vtx_results.summary(RES), name="value").to_frame())
    print("Tumour cycle diagnostics:", vtx_results.tumour_cycle_diagnostics(RES))
    fig = vtx_results.plot_overview(RES)
    fig.savefig(RUN_DIR / "overview.png", dpi=150)
    plt.show()
    display(vtx_results.choice_counts(RES))


In [ ]:
#@title Keep the results: Drive copy and/or download
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
result_zip = RUN_DIR.with_name(RUN_DIR.name + "_results.zip")
if BACKUP_ROOT and RUN_DIR.exists():
    print("Copy in Google Drive:", vtx_colab.backup_run(RUN_DIR, BACKUP_ROOT))
if DOWNLOAD_RESULTS and result_zip.is_file() and IN_COLAB:
    from google.colab import files
    files.download(str(result_zip))
    print("Download requested. Check your browser's Downloads folder before closing this session.")
elif not result_zip.is_file():
    print("No result ZIP exists yet. Start the simulation first, then rerun this cell.")
else:
    print("Results ZIP:", result_zip)


## Method and references
Jev chooses among the engine's licensed actions through the TypeSafe Choice API
([System One API](https://docs.typesafe.ai/api/system-one), [Choice schema](https://docs.typesafe.ai/primitives/choice),
[pinned models](https://docs.typesafe.ai/models)). The selected action is used directly; returned probabilities rank
behaviours and are not reaction rates. The numerical engine controls movement, contacts, clocks, fields,
cell-cycle progression and outcomes. Parameter changes are saved in `effective_contract.json` beside the original
contract fingerprint. To revisit saved results later without re-running anything, use
`Part4_Explore_Saved_Results.ipynb`.
